# Analyze Panoramic Image Sequences with Gemini 3.5 Flash

This notebook demonstrates how to query panoramic Street View imagery directly from BigQuery (`imagery_insights___us.pano_observations_latest`), visually display the images, and analyze sequences using **Gemini 3.5 Flash** to identify logistical barriers (driveway gates, fences, roadblocks, and signage).

In [ ]:
# @title 1. Installation & Environment Setup
# @markdown Install required libraries and configure project settings.

!pip install -q --upgrade google-genai google-cloud-bigquery google-cloud-storage pandas pillow

import os
import google.auth
from google.cloud import bigquery, storage
from google import genai
from google.genai import types
from IPython.display import Image, display, Markdown
import pandas as pd

# Project and Data Configuration
project_id = "YOUR_PROJECT_ID" #@param {type:"string"}
dataset_id = "imagery_insights___us" #@param {type:"string"}
table_name = "pano_observations_latest" #@param {type:"string"}
limit_count = 10 #@param {type:"integer"}

# Vertex AI Model Configuration
location = "global" #@param {type:"string"}
model_name = "gemini-3.5-flash" #@param {type:"string"}

# Auto-detect GCP project if placeholder is unchanged
if project_id == "YOUR_PROJECT_ID":
    try:
        _, auth_project = google.auth.default()
        if auth_project:
            project_id = auth_project
            print(f"[INFO] Auto-detected Google Cloud Project: {project_id}")
    except Exception:
        pass

print(f"[INFO] Configured project: '{project_id}', table: '{dataset_id}.{table_name}', limit: {limit_count}")

In [ ]:
# @title 2. Data Extraction & Image Display
# @markdown Fetch panoramic observations from BigQuery and display the retrieved images.

client_bq = bigquery.Client(project=project_id)
client_storage = storage.Client(project=project_id)

query = f"""
    SELECT
        observation_id,
        capture_id,
        gcs_uri,
        capture_time
    FROM
        `{project_id}`.`{dataset_id}`.`{table_name}`
    LIMIT {limit_count}
"""

print(f"Executing BigQuery query on `{project_id}`.`{dataset_id}`.`{table_name}`...")
try:
    df_results = client_bq.query(query).to_dataframe()
    if not df_results.empty:
        print(f"[SUCCESS] Retrieved {len(df_results)} panoramic observations.")
        display(df_results.head())
        
        print("\nVisualizing retrieved panoramic images:")
        for idx, row in df_results.iterrows():
            gcs_uri = row['gcs_uri']
            display(Markdown(f"**Observation ID**: `{row['observation_id']}` | **URI**: `{gcs_uri}`"))
            try:
                if gcs_uri.startswith("gs://"):
                    path_parts = gcs_uri[5:].split('/', 1)
                    bucket = client_storage.bucket(path_parts[0])
                    blob = bucket.blob(path_parts[1])
                    display(Image(data=blob.download_as_bytes(), width=450))
                elif gcs_uri.startswith("http"):
                    display(Image(url=gcs_uri, width=450))
            except Exception as e:
                print(f"[WARN] Could not render image thumbnail for {gcs_uri}: {e}")
    else:
        print("[WARN] Query returned 0 rows. Please verify table contents.")
except Exception as e:
    print(f"[ERROR] BigQuery query failed: {e}")
    df_results = pd.DataFrame()

In [ ]:
# @title 3. Multimodal Analysis with Vertex AI Gemini
# @markdown This cell passes panoramic images to Gemini 3.5 Flash to identify logistical features.

client_genai = genai.Client(vertexai=True, project=project_id, location=location)

analysis_prompt = """
Analyze this street view image for a logistics driver context.
Identify and describe the following features:
- **Obstructions to driveway entry / fences**: Describe any physical barriers.
- **Obstructions to street entry / Roadblocks**: Note any road blocks.
- **Signage**: Look for vehicle size/weight restrictions.
- **Gated entry**: State if a gate is present and its status (open/closed).

Provide a structured summary. If a feature is not present, state 'None'.
"""

vertex_responses = []

if 'df_results' in locals() and not df_results.empty:
    for idx, row in df_results.iterrows():
        gcs_uri = row['gcs_uri']
        obs_id = row['observation_id']
        print(f"Processing Observation: {obs_id}...")
        
        content_parts = [
            types.Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg"),
            types.Part.from_text(text=analysis_prompt)
        ]

        try:
            response = client_genai.models.generate_content(
                model=model_name,
                contents=content_parts,
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_HIGH
                )
            )
            vertex_responses.append({'observation_id': obs_id, 'gcs_uri': gcs_uri, 'response': response.text})
        except Exception as e:
            print(f"[ERROR] Failed to process {obs_id}: {e}")

    print("\n[SUCCESS] Sequence analysis complete.")
else:
    print("[WARN] No data available for analysis.")

In [ ]:
# @title 4. Final Report & Insights
# @markdown Display the AI-generated insights for each observation.

if vertex_responses:
    display(Markdown("## Fleet Logistics Insight Report"))
    for entry in vertex_responses:
        display(Markdown(f"### Observation ID: `{entry['observation_id']}`"))
        display(Markdown(f"**GCS URI**: `{entry['gcs_uri']}`"))
        display(Markdown("**AI Analysis Results:**"))
        display(Markdown(entry['response']))
        display(Markdown("---"))
else:
    print("No results to display.")